# Evaluating Retrieval-Augmented Generation (RAG) Systems
# with LLMs

## MSA 8700 — Module 7: RAG Evaluation

---

## Learning Objectives

By the end of this lecture, you will be able to:

1. Explain the **evaluation pipeline** for RAG systems using curated test sets
2. Compute and interpret **token-based metrics** (Exact Match, Token F1, BLEU, ROUGE, METEOR)
3. Identify the **limitations** of token-based metrics for measuring faithfulness, accuracy, fluency, and hallucinations
4. Describe how **LLMs are used as judges** and construct evaluation prompts
5. Compare **general-purpose vs. specialized** LLM evaluators
6. Explain the **RAGAS** and **ARES** frameworks in detail
7. Evaluate the evaluation frameworks themselves using **distributions and correlations**

## Test OpenAI

In [1]:
import os
home_dir = os.getenv("HOME", "~")
### change the file name where you store your API key
api_key = open(f"{home_dir}/.secrets/openai_pmolnar_gsu_edu_msa8700.apikey", "r").read().strip()
print(len(api_key), api_key[:3])
os.environ['OPENAI_API_KEY'] = api_key

164 sk-


In [13]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

# 1. Create an instance of ChatOpenAI
# It automatically uses the OPENAI_API_KEY environment variable.
# You can also specify the model explicitly, e.g., model="gpt-4"
chat = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.7)

# 2. Define your messages
messages = [ 
    SystemMessage(content="You are a helpful assistant that translates English to French."),
    HumanMessage(content="I love programming.")
   
]

# 3. Invoke the model to get a response
response = chat.invoke(messages)

# 4. Extract and print the content
print(response.content)

J'adore programmer.


---

# Part 1: The RAG Evaluation Pipeline

## The Core Problem

You have built a RAG system. It retrieves documents and generates answers. **But how do you know if the answers are any good?**

The standard approach follows a systematic pipeline:

```
Step 1: Create a curated Question/Answer test set (ground truth)
            ↓
Step 2: Feed the questions to your RAG system → get RAG responses
            ↓
Step 3: Compare RAG responses with ground truth answers
            ↓
Step 4: Calculate evaluation metrics
            ↓
Step 5: Analyze results and identify areas for improvement
```

## The Test Set

A curated test set consists of:

| Component | Description | Example |
|-----------|-------------|--------|
| **Question** | A question a user might ask | "When was Python created?" |
| **Ground Truth Answer** | The correct, verified answer | "Python was created in 1989 by Guido van Rossum" |
| **Retrieved Context** (optional) | The documents the RAG system retrieved | Documents about Python history |
| **RAG Response** | What your RAG system actually generated | "Python was developed in 1989" |

The fundamental question is: **How similar is the RAG response to the ground truth answer?** And beyond similarity — is the response *correct*, *relevant*, *faithful to sources*, and *free of hallucinations*?

## Setting Up Our Sample Data

Let's create a sample test set that we'll use throughout this lecture to demonstrate different evaluation approaches.

In [14]:
# Sample RAG evaluation test set
test_set = [
    {
        "question": "When was the Eiffel Tower built?",
        "ground_truth": "The Eiffel Tower was built in 1889 for the Paris Exposition",
        "rag_response": "The Eiffel Tower was constructed in 1889 for the Paris World Fair",
        "context": "The Eiffel Tower was built in 1889 by Gustave Eiffel for the Paris Exposition."
    },
    {
        "question": "Who invented the telephone?",
        "ground_truth": "The telephone was invented in 1876 by Alexander Graham Bell",
        "rag_response": "Alexander Graham Bell invented the telephone in 1876 and his assistant Margaret Thomson conducted crucial experiments",
        "context": "Alexander Graham Bell patented the telephone in 1876."
    },
    {
        "question": "What is the capital of Canada?",
        "ground_truth": "Ottawa is the capital of Canada. It is located in Ontario.",
        "rag_response": "Ottawa is the capital of Canada. It was founded in 1826 by French explorer Jean-Baptiste Rousseau.",
        "context": "Ottawa is the capital city of Canada, located in Ontario."
    },
    {
        "question": "Who was the first President of the United States?",
        "ground_truth": "George Washington was the first President of the United States, serving from 1789 to 1797",
        "rag_response": "George Washington was the first Emperor of the United States, serving from 1789 to 1797",
        "context": "George Washington served as the first President of the United States from 1789 to 1797."
    },
]

print(f"Test set contains {len(test_set)} question-answer pairs.")
for i, item in enumerate(test_set):
    print(f"\n--- Example {i+1} ---")
    print(f"  Q: {item['question']}")
    print(f"  Ground Truth: {item['ground_truth']}")
    print(f"  RAG Response: {item['rag_response']}")

Test set contains 4 question-answer pairs.

--- Example 1 ---
  Q: When was the Eiffel Tower built?
  Ground Truth: The Eiffel Tower was built in 1889 for the Paris Exposition
  RAG Response: The Eiffel Tower was constructed in 1889 for the Paris World Fair

--- Example 2 ---
  Q: Who invented the telephone?
  Ground Truth: The telephone was invented in 1876 by Alexander Graham Bell
  RAG Response: Alexander Graham Bell invented the telephone in 1876 and his assistant Margaret Thomson conducted crucial experiments

--- Example 3 ---
  Q: What is the capital of Canada?
  Ground Truth: Ottawa is the capital of Canada. It is located in Ontario.
  RAG Response: Ottawa is the capital of Canada. It was founded in 1826 by French explorer Jean-Baptiste Rousseau.

--- Example 4 ---
  Q: Who was the first President of the United States?
  Ground Truth: George Washington was the first President of the United States, serving from 1789 to 1797
  RAG Response: George Washington was the first Emperor

---

# Part 2: Token-Based Evaluation Metrics

Token-based metrics measure **surface-level similarity** between the RAG response and the ground truth answer. They answer one question:

> *"How similar is this answer to the reference answer in terms of word/token overlap?"*

## Quick Comparison Table

| Metric | Core Idea | Precision vs Recall | Synonyms? | Word Order? | Best For |
|--------|-----------|---------------------|-----------|-------------|----------|
| **Exact Match** | Binary match | N/A | No | Yes | Exact factual answers |
| **Token F1** | Set overlap | Balanced | No | No | Short QA |
| **BLEU** | N-gram precision | Precision-focused | No | Via n-grams | Translation, phrase matching |
| **ROUGE-1** | Unigram overlap | Recall-focused | No | No | Summarization, abstractive QA |
| **ROUGE-L** | Longest common subsequence | Balanced | No | Yes (LCS) | Any task (order-aware) |
| **METEOR** | Matched words + ordering | Balanced (R-favored) | Yes | Yes (penalty) | Translation, highest human correlation |

## 2.1 Exact Match (EM)

The simplest metric: **Is the generated answer exactly the same as the reference?**

$$\text{EM} = \begin{cases} 1 & \text{if } \text{normalize}(\text{response}) = \text{normalize}(\text{reference}) \\ 0 & \text{otherwise} \end{cases}$$

- **Use when:** Questions with a single correct answer (dates, names, numbers)
- **Limitation:** Too strict for paraphrases — "Python was created in 1989" vs. "1989 was when Python was created" scores 0

In [1]:
def exact_match(reference, candidate):
    """Binary metric: 1 if exact match after normalization, 0 otherwise."""
    return 1.0 if reference.lower().strip() == candidate.lower().strip() else 0.0

# Demonstrate
print("=" * 70)
print("EXACT MATCH EXAMPLES")
print("=" * 70)

examples = [
    ("Python was created in 1989", "Python was created in 1989"),
    ("Python was created in 1989", "python was created in 1989"),
    ("Python was created in 1989", "Python was developed in 1989"),
    ("1989", "1989"),
    ("1989", "1990"),
]

for ref, cand in examples:
    score = exact_match(ref, cand)
    print(f"\n  Ref:  '{ref}'")
    print(f"  Cand: '{cand}'")
    print(f"  EM = {score:.1f}  {'✓' if score == 1.0 else '✗'}")

EXACT MATCH EXAMPLES

  Ref:  'Python was created in 1989'
  Cand: 'Python was created in 1989'
  EM = 1.0  ✓

  Ref:  'Python was created in 1989'
  Cand: 'python was created in 1989'
  EM = 1.0  ✓

  Ref:  'Python was created in 1989'
  Cand: 'Python was developed in 1989'
  EM = 0.0  ✗

  Ref:  '1989'
  Cand: '1989'
  EM = 1.0  ✓

  Ref:  '1989'
  Cand: '1990'
  EM = 0.0  ✗


## 2.2 Token F1 Score

Treats both reference and candidate as **sets of tokens** and computes:

$$\text{Precision} = \frac{|\text{common tokens}|}{|\text{candidate tokens}|}$$

$$\text{Recall} = \frac{|\text{common tokens}|}{|\text{reference tokens}|}$$

$$\text{F1} = \frac{2 \times \text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

- **Precision** = "Of the words in the candidate, how many are in the reference?" (did the model avoid adding wrong words?)
- **Recall** = "Of the words in the reference, how many appear in the candidate?" (did the model capture all key info?)
- **Limitation:** Ignores word order — "Alice gave book to Bob" and "Bob gave book to Alice" get F1 = 1.0

In [3]:
def tokenize(text):
    """Simple whitespace tokenization with lowercasing."""
    return text.lower().split()

def token_f1(reference, candidate):
    """Compute precision, recall, and F1 at the token (set) level."""
    ref_set = set(tokenize(reference))
    cand_set = set(tokenize(candidate))
    
    common = ref_set & cand_set
    
    precision = len(common) / len(cand_set) if cand_set else 0.0
    recall = len(common) / len(ref_set) if ref_set else 0.0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return f1, precision, recall

# Walkthrough example
# ref = "The Eiffel Tower was built in 1889 for the Paris Exposition"
# cand = "The Eiffel Tower was constructed in 1889 for the Paris World Fair"
ref = "The man chased the dog"
cand = "The dog chased the man"

ref_tokens = set(tokenize(ref))
cand_tokens = set(tokenize(cand))
common = ref_tokens & cand_tokens

print("TOKEN F1 — STEP BY STEP")
print("=" * 60)
print(f"Reference tokens: {ref_tokens}")
print(f"Candidate tokens: {cand_tokens}")
print(f"Common tokens:    {common}")
print(f"\nPrecision = {len(common)}/{len(cand_tokens)} = {len(common)/len(cand_tokens):.3f}")
print(f"Recall    = {len(common)}/{len(ref_tokens)} = {len(common)/len(ref_tokens):.3f}")

f1, p, r = token_f1(ref, cand)
print(f"F1        = 2 * {p:.3f} * {r:.3f} / ({p:.3f} + {r:.3f}) = {f1:.3f}")

TOKEN F1 — STEP BY STEP
Reference tokens: {'chased', 'the', 'man', 'dog'}
Candidate tokens: {'chased', 'the', 'man', 'dog'}
Common tokens:    {'dog', 'chased', 'man', 'the'}

Precision = 4/4 = 1.000
Recall    = 4/4 = 1.000
F1        = 2 * 1.000 * 1.000 / (1.000 + 1.000) = 1.000


## 2.3 BLEU Score (Bilingual Evaluation Understudy)

BLEU measures **n-gram precision**: what fraction of the candidate's n-grams appear in the reference?

$$\text{BLEU} = \text{BP} \times \exp\left(\sum_{n=1}^{N} w_n \log p_n\right)$$

Where:
- $p_n$ = precision of n-grams (fraction of candidate n-grams that appear in reference)
- $w_n$ = weights (typically $1/N$ for uniform weighting)
- $\text{BP}$ = brevity penalty (penalizes if candidate is shorter than reference)

### Key characteristics:
- **Precision-focused** — rewards exact phrase matches
- **Uses geometric mean** — if any n-gram precision is 0, BLEU = 0
- **Brevity penalty** — discourages overly short answers
- **Harsh on paraphrasing** — different word choices are penalized

In [4]:
from collections import Counter
import math

def get_ngrams(tokens, n):
    """Extract n-grams from a list of tokens."""
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

def bleu_score(reference, candidate, max_n=4):
    """Compute BLEU score: geometric mean of n-gram precisions with brevity penalty."""
    ref_tokens = tokenize(reference)
    cand_tokens = tokenize(candidate)
    
    if len(cand_tokens) == 0:
        return 0.0
    
    precisions = []
    for n in range(1, max_n + 1):
        ref_ngrams = Counter(get_ngrams(ref_tokens, n))
        cand_ngrams = Counter(get_ngrams(cand_tokens, n))
        matching = sum((cand_ngrams & ref_ngrams).values())
        total = sum(cand_ngrams.values())
        precisions.append(matching / total if total > 0 else 0.0)
    
    # Geometric mean
    if all(p > 0 for p in precisions):
        bleu = math.exp(sum(math.log(p) for p in precisions) / len(precisions))
    else:
        bleu = 0.0
    
    # Brevity penalty
    ratio = len(cand_tokens) / len(ref_tokens) if len(ref_tokens) > 0 else 0
    bp = math.exp(1 - 1/ratio) if 0 < ratio < 1 else 1.0
    
    return bleu * bp

# Step-by-step walkthrough
ref = "the cat is on the mat"
cand = "the cat on the mat"

print("BLEU SCORE — STEP BY STEP")
print("=" * 70)
print(f"Reference: '{ref}'")
print(f"Candidate: '{cand}'  (missing 'is')")

ref_tok = tokenize(ref)
cand_tok = tokenize(cand)

for n in range(1, 5):
    ref_ng = Counter(get_ngrams(ref_tok, n))
    cand_ng = Counter(get_ngrams(cand_tok, n))
    matching = sum((cand_ng & ref_ng).values())
    total = sum(cand_ng.values())
    prec = matching / total if total > 0 else 0.0
    print(f"\n  {n}-gram precision: {matching}/{total} = {prec:.4f}")
    print(f"    Ref {n}-grams: {list(ref_ng.keys())}")
    print(f"    Cand {n}-grams: {list(cand_ng.keys())}")

score = bleu_score(ref, cand)
print(f"\n  BLEU-4 = {score:.4f}")
print(f"\n  NOTE: Because 4-gram precision is 0, geometric mean → BLEU = 0")
print(f"  This shows BLEU is HARSH — one missing word can kill the score!")

BLEU SCORE — STEP BY STEP
Reference: 'the cat is on the mat'
Candidate: 'the cat on the mat'  (missing 'is')

  1-gram precision: 5/5 = 1.0000
    Ref 1-grams: [('the',), ('cat',), ('is',), ('on',), ('mat',)]
    Cand 1-grams: [('the',), ('cat',), ('on',), ('mat',)]

  2-gram precision: 3/4 = 0.7500
    Ref 2-grams: [('the', 'cat'), ('cat', 'is'), ('is', 'on'), ('on', 'the'), ('the', 'mat')]
    Cand 2-grams: [('the', 'cat'), ('cat', 'on'), ('on', 'the'), ('the', 'mat')]

  3-gram precision: 1/3 = 0.3333
    Ref 3-grams: [('the', 'cat', 'is'), ('cat', 'is', 'on'), ('is', 'on', 'the'), ('on', 'the', 'mat')]
    Cand 3-grams: [('the', 'cat', 'on'), ('cat', 'on', 'the'), ('on', 'the', 'mat')]

  4-gram precision: 0/2 = 0.0000
    Ref 4-grams: [('the', 'cat', 'is', 'on'), ('cat', 'is', 'on', 'the'), ('is', 'on', 'the', 'mat')]
    Cand 4-grams: [('the', 'cat', 'on', 'the'), ('cat', 'on', 'the', 'mat')]

  BLEU-4 = 0.0000

  NOTE: Because 4-gram precision is 0, geometric mean → BLEU = 0
  T

## 2.4 ROUGE (Recall-Oriented Understudy for Gisting Evaluation)

ROUGE is **recall-focused** — it asks: "Of the words/n-grams in the reference, how many appear in the candidate?"

### ROUGE-1 (Unigram Overlap)

$$\text{ROUGE-1 Recall} = \frac{\text{matching unigrams}}{\text{total reference unigrams}}$$

$$\text{ROUGE-1 Precision} = \frac{\text{matching unigrams}}{\text{total candidate unigrams}}$$

### ROUGE-L (Longest Common Subsequence)

Uses the **Longest Common Subsequence (LCS)** — the longest sequence of words that appear in both texts in order (but not necessarily contiguously).

$$\text{ROUGE-L Recall} = \frac{\text{LCS length}}{\text{reference length}}$$

ROUGE-L is **order-aware** — if words appear in different order, the LCS is shorter.

### Key insight: ROUGE-1 vs ROUGE-L
- **ROUGE-1** treats text as a **bag of words** — ignores order
- **ROUGE-L** preserves **word order** via LCS
- This matters when word order changes meaning ("Alice gave Bob" vs. "Bob gave Alice")

In [5]:
def rouge_n(reference, candidate, n=1):
    """ROUGE-N: n-gram overlap with precision, recall, and F1."""
    ref_tokens = tokenize(reference)
    cand_tokens = tokenize(candidate)
    
    ref_ngrams = Counter(get_ngrams(ref_tokens, n))
    cand_ngrams = Counter(get_ngrams(cand_tokens, n))
    
    matching = sum((cand_ngrams & ref_ngrams).values())
    
    recall = matching / sum(ref_ngrams.values()) if sum(ref_ngrams.values()) > 0 else 0.0
    precision = matching / sum(cand_ngrams.values()) if sum(cand_ngrams.values()) > 0 else 0.0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return f1, precision, recall

def rouge_l(reference, candidate):
    """ROUGE-L: Longest Common Subsequence with dynamic programming."""
    ref_tokens = tokenize(reference)
    cand_tokens = tokenize(candidate)
    
    m, n = len(ref_tokens), len(cand_tokens)
    lcs = [[0] * (n + 1) for _ in range(m + 1)]
    
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if ref_tokens[i-1] == cand_tokens[j-1]:
                lcs[i][j] = lcs[i-1][j-1] + 1
            else:
                lcs[i][j] = max(lcs[i-1][j], lcs[i][j-1])
    
    lcs_length = lcs[m][n]
    precision = lcs_length / len(cand_tokens) if len(cand_tokens) > 0 else 0.0
    recall = lcs_length / len(ref_tokens) if len(ref_tokens) > 0 else 0.0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return f1, precision, recall, lcs_length

# ROUGE-1 walkthrough
ref = "the cat is on the mat"
cand = "the cat on the mat"

print("ROUGE — STEP BY STEP")
print("=" * 70)
print(f"Reference: '{ref}'")
print(f"Candidate: '{cand}'")

# ROUGE-1
ref_counter = Counter(tokenize(ref))
cand_counter = Counter(tokenize(cand))
common = cand_counter & ref_counter
print(f"\nROUGE-1 (Unigram Overlap):")
print(f"  Reference words: {dict(ref_counter)}")
print(f"  Candidate words: {dict(cand_counter)}")
print(f"  Common words:    {dict(common)} → total = {sum(common.values())}")

f1_1, p1, r1 = rouge_n(ref, cand, n=1)
print(f"  Recall    = {sum(common.values())}/{sum(ref_counter.values())} = {r1:.4f}")
print(f"  Precision = {sum(common.values())}/{sum(cand_counter.values())} = {p1:.4f}")
print(f"  F1        = {f1_1:.4f}")

# ROUGE-L
f1_l, pl, rl, lcs_len = rouge_l(ref, cand)
print(f"\nROUGE-L (Longest Common Subsequence):")
print(f"  LCS length = {lcs_len}")
print(f"  Recall     = {lcs_len}/{len(tokenize(ref))} = {rl:.4f}")
print(f"  Precision  = {lcs_len}/{len(tokenize(cand))} = {pl:.4f}")
print(f"  F1         = {f1_l:.4f}")

ROUGE — STEP BY STEP
Reference: 'the cat is on the mat'
Candidate: 'the cat on the mat'

ROUGE-1 (Unigram Overlap):
  Reference words: {'the': 2, 'cat': 1, 'is': 1, 'on': 1, 'mat': 1}
  Candidate words: {'the': 2, 'cat': 1, 'on': 1, 'mat': 1}
  Common words:    {'the': 2, 'cat': 1, 'on': 1, 'mat': 1} → total = 5
  Recall    = 5/6 = 0.8333
  Precision = 5/5 = 1.0000
  F1        = 0.9091

ROUGE-L (Longest Common Subsequence):
  LCS length = 5
  Recall     = 5/6 = 0.8333
  Precision  = 5/5 = 1.0000
  F1         = 0.9091


### Why Word Order Matters: ROUGE-1 vs ROUGE-L

Consider a case where the same words appear but the **meaning is completely different**:

In [7]:
# Word order changes meaning!
ref = "Alice gave a book to Bob"
cand = "Bob gave a book to Alice"

print("WORD ORDER MATTERS")
print("=" * 60)
print(f"Reference: '{ref}'")
print(f"Candidate: '{cand}'")
print(f"\nMeaning is COMPLETELY DIFFERENT!")

f1_1, _, _ = rouge_n(ref, cand, n=1)
f1_l, _, _, _ = rouge_l(ref, cand)

print(f"\n  ROUGE-1 F1: {f1_1:.3f}  ← Sees all words match → FAILS")
print(f"  ROUGE-L F1: {f1_l:.3f}  ← LCS catches some scrambling → PARTIAL")
print(f"\n  ROUGE-1 completely misses the meaning change!")
print(f"  ROUGE-L is better but still gives a high score.")

WORD ORDER MATTERS
Reference: 'Alice gave a book to Bob'
Candidate: 'Bob gave a book to Alice'

Meaning is COMPLETELY DIFFERENT!

  ROUGE-1 F1: 1.000  ← Sees all words match → FAILS
  ROUGE-L F1: 0.667  ← LCS catches some scrambling → PARTIAL

  ROUGE-1 completely misses the meaning change!
  ROUGE-L is better but still gives a high score.


## 2.5 METEOR (Metric for Evaluation of Translation with Explicit ORdering)

METEOR improves on BLEU and ROUGE by:
1. **Matching synonyms** ("built" ≈ "constructed")
2. **Matching stems** ("running" ≈ "runs")
3. **Penalizing word order scrambling** via a fragmentation penalty

METEOR computes a weighted harmonic mean that **emphasizes recall** ($\beta = 3$):

$$F_{\text{METEOR}} = \frac{(1 + \beta^2) \cdot P \cdot R}{\beta^2 \cdot P + R}$$

Then applies a penalty for fragmented matching (words out of order).

- **Best correlation with human judgment** among token-based metrics
- **Computationally heavier** (requires synonym database)

In [6]:
def simplified_meteor(reference, candidate):
    """Simplified METEOR: unigram matches with recall-weighted F-score.
    Full METEOR also includes stemming, synonymy, and word order penalty."""
    ref_tokens = tokenize(reference)
    cand_tokens = tokenize(candidate)
    
    ref_counter = Counter(ref_tokens)
    cand_counter = Counter(cand_tokens)
    
    matches = sum((cand_counter & ref_counter).values())
    
    precision = matches / len(cand_tokens) if len(cand_tokens) > 0 else 0.0
    recall = matches / len(ref_tokens) if len(ref_tokens) > 0 else 0.0
    
    if precision + recall == 0:
        return 0.0
    
    # beta=3 means recall is weighted 3x more than precision
    beta = 3.0
    f_score = (1 + beta**2) * (precision * recall) / ((beta**2 * precision) + recall)
    
    return f_score

# Demonstrate METEOR's advantage with synonyms
print("METEOR — SYNONYM HANDLING")
print("=" * 60)

ref = "The quick brown fox jumps"
cand = "The fast brown fox leaps"

print(f"Reference: '{ref}'")
print(f"Candidate: '{cand}'")
print(f"  'quick' → 'fast'  (synonym)")
print(f"  'jumps' → 'leaps' (synonym)")

# Without synonyms (our simplified version)
score_simple = simplified_meteor(ref, cand)
print(f"\n  Simplified METEOR (exact match only): {score_simple:.3f}")
print(f"  Full METEOR (with synonyms):           ~0.85-0.95")
print(f"\n  Full METEOR recognizes synonyms and gives higher score!")

METEOR — SYNONYM HANDLING
Reference: 'The quick brown fox jumps'
Candidate: 'The fast brown fox leaps'
  'quick' → 'fast'  (synonym)
  'jumps' → 'leaps' (synonym)

  Simplified METEOR (exact match only): 0.600
  Full METEOR (with synonyms):           ~0.85-0.95

  Full METEOR recognizes synonyms and gives higher score!


## 2.6 Computing All Metrics on Our Test Set

Let's compute all token-based metrics on our sample test set and see how they compare:

In [9]:
print("COMPREHENSIVE METRIC COMPARISON ON TEST SET")
print("=" * 90)

for i, item in enumerate(test_set):
    ref = item["ground_truth"]
    cand = item["rag_response"]
    
    em = exact_match(ref, cand)
    f1, p, r = token_f1(ref, cand)
    bleu = bleu_score(ref, cand)
    r1_f1, _, _ = rouge_n(ref, cand, n=1)
    r2_f1, _, _ = rouge_n(ref, cand, n=2)
    rl_f1, _, _, _ = rouge_l(ref, cand)
    meteor = simplified_meteor(ref, cand)
    
    print(f"\n{'─' * 90}")
    print(f"Example {i+1}: {item['question']}")
    print(f"  Ref:  {ref}")
    print(f"  Cand: {cand}")
    print(f"  {'─' * 50}")
    print(f"  {'Exact Match':<20} {em:>8.3f}")
    print(f"  {'Token F1':<20} {f1:>8.3f}  (P={p:.3f}, R={r:.3f})")
    print(f"  {'BLEU-4':<20} {bleu:>8.3f}")
    print(f"  {'ROUGE-1 F1':<20} {r1_f1:>8.3f}")
    print(f"  {'ROUGE-2 F1':<20} {r2_f1:>8.3f}")
    print(f"  {'ROUGE-L F1':<20} {rl_f1:>8.3f}")
    print(f"  {'METEOR (simplified)':<20} {meteor:>8.3f}")

COMPREHENSIVE METRIC COMPARISON ON TEST SET

──────────────────────────────────────────────────────────────────────────────────────────
Example 1: When was the Eiffel Tower built?
  Ref:  The Eiffel Tower was built in 1889 for the Paris Exposition
  Cand: The Eiffel Tower was constructed in 1889 for the Paris World Fair
  ──────────────────────────────────────────────────
  Exact Match             0.000
  Token F1                0.762  (P=0.727, R=0.800)
  BLEU-4                  0.531
  ROUGE-1 F1              0.783
  ROUGE-2 F1              0.667
  ROUGE-L F1              0.783
  METEOR (simplified)     0.811

──────────────────────────────────────────────────────────────────────────────────────────
Example 2: Who invented the telephone?
  Ref:  The telephone was invented in 1876 by Alexander Graham Bell
  Cand: Alexander Graham Bell invented the telephone in 1876 and his assistant Margaret Thomson conducted crucial experiments
  ──────────────────────────────────────────────────
  E

### Discussion: What Do These Scores Tell Us?

Look at **Example 4** (Washington/Emperor). The metrics show **high similarity** (0.9+) because almost all words match. But the answer says "Emperor" instead of "President" — a **critical factual error**. The metrics completely miss this!

And **Example 3** (Ottawa/Rousseau). The metrics show decent scores, but the generated answer contains a **fabricated claim** about a fictional French explorer. The metrics can't distinguish real facts from hallucinated ones.

This leads us to a fundamental truth about token-based metrics...

---

# Part 3: Limitations of Token-Based Metrics

Token-based metrics answer **one question**:

> *"How similar is this answer to the reference answer?"*

They **cannot** answer these critical questions:

| Question | Dimension | Can Token Metrics Measure? |
|----------|-----------|---------------------------|
| "Is the answer relevant to the question?" | **Relevancy** | No |
| "Are the facts in the answer true?" | **Truthfulness / Accuracy** | No |
| "Does the answer read naturally?" | **Fluency** | No |
| "Did the model make up false information?" | **Hallucinations** | No |

### Why? Because token metrics are:

| Property | Implication |
|----------|-------------|
| **Reference-dependent** | Need a gold-standard answer to compare against |
| **Surface-level** | Work at word/phrase level, not semantic level |
| **Non-factual** | Don't check facts against reality |
| **Non-linguistic** | Don't evaluate grammar or fluency |
| **Similarity-focused** | Measure how "similar" not how "correct" |

## 3.1 Failure Case: Truthfulness

A factually **wrong** answer can score 0.95+ because only one word is different:

In [10]:
print("TRUTHFULNESS FAILURE")
print("=" * 70)

ref = "George Washington was the first President of the United States serving from 1789 to 1797"
cand = "George Washington was the first Emperor of the United States serving from 1789 to 1797"

print(f"Reference: ...first President of the United States...")
print(f"Generated: ...first Emperor of the United States...")
print(f"\n  One word changed: 'President' → 'Emperor'")
print(f"  FACTUAL STATUS: COMPLETELY WRONG")

r1, _, _ = rouge_n(ref, cand, n=1)
f1, _, _ = token_f1(ref, cand)
print(f"\n  ROUGE-1 F1: {r1:.3f}  ← 'Nearly perfect!'")
print(f"  Token F1:   {f1:.3f}  ← 'Excellent match!'")
print(f"\n  Metrics say: Great answer")
print(f"  Reality:     Factually WRONG")

TRUTHFULNESS FAILURE
Reference: ...first President of the United States...
Generated: ...first Emperor of the United States...

  One word changed: 'President' → 'Emperor'
  FACTUAL STATUS: COMPLETELY WRONG

  ROUGE-1 F1: 0.933  ← 'Nearly perfect!'
  Token F1:   0.929  ← 'Excellent match!'

  Metrics say: Great answer
  Reality:     Factually WRONG


## 3.2 Failure Case: Fluency

A **scrambled, ungrammatical** answer can score 1.0:

In [11]:
print("FLUENCY FAILURE")
print("=" * 70)

ref = "Photosynthesis is the process by which plants convert sunlight into chemical energy"
cand = "Sunlight into chemical energy plants convert process is photosynthesis"

print(f"Reference: {ref}")
print(f"Generated: {cand}")
print(f"\n  Same words, completely scrambled!")
print(f"  A human would rate fluency: 0/10")

r1, _, _ = rouge_n(ref, cand, n=1)
f1, _, _ = token_f1(ref, cand)
print(f"\n  ROUGE-1 F1: {r1:.3f}  ← 'Perfect match!'")
print(f"  Token F1:   {f1:.3f}  ← 'Perfect match!'")
print(f"\n  Metrics say: Perfect answer")
print(f"  Reality:     Unreadable word jumble")

FLUENCY FAILURE
Reference: Photosynthesis is the process by which plants convert sunlight into chemical energy
Generated: Sunlight into chemical energy plants convert process is photosynthesis

  Same words, completely scrambled!
  A human would rate fluency: 0/10

  ROUGE-1 F1: 0.857  ← 'Perfect match!'
  Token F1:   0.857  ← 'Perfect match!'

  Metrics say: Perfect answer
  Reality:     Unreadable word jumble


## 3.3 Failure Case: Hallucinations

A hallucinated answer with fabricated facts can score well:

In [12]:
print("HALLUCINATION FAILURE")
print("=" * 70)

ref = "Ottawa is the capital of Canada. It is located in Ontario."
cand = "Ottawa is the capital of Canada. It was founded in 1826 by French explorer Jean-Baptiste Rousseau."

print(f"Reference: {ref}")
print(f"Generated: {cand}")
print(f"\n  Fact check:")
print(f"  ✓ 'Ottawa is the capital of Canada' — CORRECT")
print(f"  ✗ 'Founded in 1826 by Jean-Baptiste Rousseau' — FABRICATED!")

r1, _, _ = rouge_n(ref, cand, n=1)
f1, _, _ = token_f1(ref, cand)
print(f"\n  ROUGE-1 F1: {r1:.3f}")
print(f"  Token F1:   {f1:.3f}")
print(f"\n  Metrics say: Decent answer")
print(f"  Reality:     Contains HALLUCINATED facts")
print(f"\n  The hallucination is INVISIBLE to token metrics!")

HALLUCINATION FAILURE
Reference: Ottawa is the capital of Canada. It is located in Ontario.
Generated: Ottawa is the capital of Canada. It was founded in 1826 by French explorer Jean-Baptiste Rousseau.

  Fact check:
  ✓ 'Ottawa is the capital of Canada' — CORRECT
  ✗ 'Founded in 1826 by Jean-Baptiste Rousseau' — FABRICATED!

  ROUGE-1 F1: 0.593
  Token F1:   0.615

  Metrics say: Decent answer
  Reality:     Contains HALLUCINATED facts

  The hallucination is INVISIBLE to token metrics!


## Summary: What Token Metrics Can and Cannot Measure

```
Token Metrics CAN Measure:        Token Metrics CANNOT Measure:
✓ Surface similarity               ✗ Question relevancy
✓ N-gram overlap                   ✗ Factual correctness
✓ Word order (ROUGE-L)             ✗ Fluency/Grammar
                                   ✗ Hallucinations

Use them for:                      Don't rely on them for:
✓ Reproducible baseline            ✗ Judging answer quality alone
✓ Automated screening              ✗ Detecting hallucinations
✓ Quick comparison                 ✗ Evaluating relevancy

ALWAYS COMBINE WITH:
• Semantic metrics (BERTScore)
• LLM-as-Judge evaluation
• Human evaluation (sampling)
```

This is where **LLMs as Judges** come in.

---

# Part 4: LLMs as Judges

## The Idea

Instead of counting word overlaps, we ask a **large language model** to evaluate the answer like a human would — assessing relevancy, accuracy, fluency, and hallucinations.

## Why LLMs Can Do What Token Metrics Cannot

| Aspect | Token Metrics | LLM Judge | Human Judge |
|--------|---------------|-----------|-------------|
| **Speed** | Instant | 1-5 sec | 5-10 min |
| **Cost** | Free | $0.001-0.01/eval | $0.50-2.00/eval |
| **Scalability** | Unlimited | Unlimited | Limited |
| **Relevancy** | No | Yes (0.80-0.90 corr.) | Yes |
| **Truthfulness** | No | Partial (0.65-0.80) | Yes |
| **Fluency** | No | Yes (0.80-0.90) | Yes |
| **Hallucinations** | No | Partial (0.60-0.75) | Yes |
| **Best Use** | Baseline | Screening | Validation |

## How It Works

1. **Construct a prompt** that asks the LLM to evaluate specific dimensions
2. **Provide context**: question, generated answer, reference (optional), source documents (optional)
3. **Request structured output** (JSON) for easy parsing
4. **Set low temperature** (0.3-0.5) for consistent scoring
5. **Validate** with human samples (target correlation > 0.7)

## 4.1 Evaluation Dimensions

### Relevancy
- **What:** Does the answer address the question that was asked?
- **LLM reliability:** High (0.80-0.90 correlation with humans)
- **Example:** Q: "When was Python created?" → A: "Python is a programming language" = **NOT RELEVANT**

### Accuracy / Truthfulness
- **What:** Are the factual claims in the answer correct?
- **LLM reliability:** Medium (0.65-0.80) — LLMs can't verify facts beyond their training data
- **Improvement:** Provide source documents in the prompt for fact-checking

### Fluency
- **What:** Is the answer well-written, grammatical, and clear?
- **LLM reliability:** High (0.80-0.90) — LLMs are excellent at language judgment

### Hallucinations
- **What:** Does the answer contain fabricated facts not supported by sources?
- **LLM reliability:** Medium-Low (0.60-0.75) — LLMs can hallucinate while detecting hallucinations!
- **Critical limitation:** LLMs can't detect hallucinations they don't know about

## 4.2 Example Evaluation Prompts

### Simple Relevancy Prompt

In [15]:
test_set

[{'question': 'When was the Eiffel Tower built?',
  'ground_truth': 'The Eiffel Tower was built in 1889 for the Paris Exposition',
  'rag_response': 'The Eiffel Tower was constructed in 1889 for the Paris World Fair',
  'context': 'The Eiffel Tower was built in 1889 by Gustave Eiffel for the Paris Exposition.'},
 {'question': 'Who invented the telephone?',
  'ground_truth': 'The telephone was invented in 1876 by Alexander Graham Bell',
  'rag_response': 'Alexander Graham Bell invented the telephone in 1876 and his assistant Margaret Thomson conducted crucial experiments',
  'context': 'Alexander Graham Bell patented the telephone in 1876.'},
 {'question': 'What is the capital of Canada?',
  'ground_truth': 'Ottawa is the capital of Canada. It is located in Ontario.',
  'rag_response': 'Ottawa is the capital of Canada. It was founded in 1826 by French explorer Jean-Baptiste Rousseau.',
  'context': 'Ottawa is the capital city of Canada, located in Ontario.'},
 {'question': 'Who was the 

In [17]:
relevancy_prompt = """
QUESTION: {question}
ANSWER: {answer}

Score relevancy (0-5):
5 = Perfectly answers the question
4 = Addresses question with minor gaps
3 = Addresses question with some gaps
2 = Partially addresses the question
1 = Barely relevant
0 = Off-topic

RESPOND: SCORE: [0-5] REASON: [one sentence]
"""

def metric_relevancy(question, answer):
    chat = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.7)

    # 2. Define your messages
    messages = [ 
        SystemMessage(content="You are a QA evaluator"),
        HumanMessage(content=relevancy_prompt.format(
            question = question,
            answer = answer
         )
        )
    ]

    # 3. Invoke the model to get a response
    response = chat.invoke(messages)
    
    # 4. Extract and print the content
    return response.content
    

In [22]:
for j in range(len(test_set)):
    q = test_set[j]["question"]
    t = test_set[j]["ground_truth"]
    a = test_set[j]["rag_response"]
    print(f"\nQ{j}: {q}")
    print(t)
    print(metric_relevancy(q, t))
    print(a)
    print(metric_relevancy(q, a))


Q0: When was the Eiffel Tower built?
The Eiffel Tower was built in 1889 for the Paris Exposition
SCORE: 5 REASON: The answer provides a direct and accurate response to the question by stating that the Eiffel Tower was built in 1889 for the Paris Exposition.
The Eiffel Tower was constructed in 1889 for the Paris World Fair
SCORE: 5 REASON: The answer provides the exact year the Eiffel Tower was built and the context in which it was constructed.

Q1: Who invented the telephone?
The telephone was invented in 1876 by Alexander Graham Bell
SCORE: 5 REASON: The answer provides a clear and concise response to the question by stating that the telephone was invented by Alexander Graham Bell in 1876.
Alexander Graham Bell invented the telephone in 1876 and his assistant Margaret Thomson conducted crucial experiments
SCORE: 4 REASON: The response provides the correct inventor of the telephone but includes unnecessary information about the assistant conducting experiments.

Q2: What is the capita

In [13]:
print("RELEVANCY EVALUATION PROMPT")
print("=" * 60)
print(relevancy_prompt.format(
    question="When was Python created?",
    answer="Python was created in 1989 by Guido van Rossum"
))
print("\nExpected LLM response:")
print('SCORE: 5 REASON: Directly answers when Python was created with the year and creator.')

RELEVANCY EVALUATION PROMPT

You are a QA evaluator.

QUESTION: When was Python created?
ANSWER: Python was created in 1989 by Guido van Rossum

Score relevancy (0-5):
5 = Perfectly answers the question
4 = Addresses question with minor gaps
3 = Addresses question with some gaps
2 = Partially addresses the question
1 = Barely relevant
0 = Off-topic

RESPOND: SCORE: [0-5] REASON: [one sentence]


Expected LLM response:
SCORE: 5 REASON: Directly answers when Python was created with the year and creator.


### Comprehensive Evaluation Prompt (All Dimensions)

In [14]:
comprehensive_prompt = """
You are an expert QA evaluator.

QUESTION: {question}
REFERENCE ANSWER: {reference}
SOURCE DOCUMENTS: {sources}
GENERATED ANSWER: {answer}

Evaluate on 4 dimensions (0-5 each):

1. RELEVANCY: Does it address the question?
   5 = Perfectly answers all parts
   0 = Off-topic

2. ACCURACY: Are facts correct? (Use sources if available)
   5 = All facts correct
   0 = Completely inaccurate

3. FLUENCY: Is it well-written and clear?
   5 = Excellent grammar, clear structure
   0 = Incomprehensible

4. HALLUCINATIONS: Any made-up facts? (5=none, 0=mostly fabricated)
   5 = No hallucinations
   0 = Almost entirely made up

Respond in JSON:
{{
  "relevancy": <0-5>,
  "accuracy": <0-5>,
  "fluency": <0-5>,
  "hallucinations": <0-5>,
  "explanation": "<brief explanation>"
}}
"""

print("COMPREHENSIVE EVALUATION PROMPT")
print("=" * 60)
# Show it filled in with our Example 3 (the hallucination case)
filled = comprehensive_prompt.format(
    question=test_set[2]["question"],
    reference=test_set[2]["ground_truth"],
    sources=test_set[2]["context"],
    answer=test_set[2]["rag_response"]
)
print(filled)

print("\nExpected LLM Judge Output:")
print("""{
  "relevancy": 4,
  "accuracy": 2,
  "fluency": 5,
  "hallucinations": 2,
  "explanation": "Answer correctly identifies Ottawa as capital but adds
   fabricated claim about founding by Jean-Baptiste Rousseau."
}""")

COMPREHENSIVE EVALUATION PROMPT

You are an expert QA evaluator.

QUESTION: What is the capital of Canada?
REFERENCE ANSWER: Ottawa is the capital of Canada. It is located in Ontario.
SOURCE DOCUMENTS: Ottawa is the capital city of Canada, located in Ontario.
GENERATED ANSWER: Ottawa is the capital of Canada. It was founded in 1826 by French explorer Jean-Baptiste Rousseau.

Evaluate on 4 dimensions (0-5 each):

1. RELEVANCY: Does it address the question?
   5 = Perfectly answers all parts
   0 = Off-topic

2. ACCURACY: Are facts correct? (Use sources if available)
   5 = All facts correct
   0 = Completely inaccurate

3. FLUENCY: Is it well-written and clear?
   5 = Excellent grammar, clear structure
   0 = Incomprehensible

4. HALLUCINATIONS: Any made-up facts? (5=none, 0=mostly fabricated)
   5 = No hallucinations
   0 = Almost entirely made up

Respond in JSON:
{
  "relevancy": <0-5>,
  "accuracy": <0-5>,
  "fluency": <0-5>,
  "hallucinations": <0-5>,
  "explanation": "<brief expla

### Hallucination Detection Prompt (Claim-by-Claim)

In [ ]:
hallucination_prompt = """
You are checking for hallucinations (made-up facts).

REFERENCE: {reference}
ANSWER: {answer}

For each claim in the answer, mark as:
  SUPPORTED    — In the reference
  UNCLEAR      — Not explicitly stated but reasonable
  UNSUPPORTED  — Not in reference, not contradicted
  HALLUCINATION — Made up, contradicted, or incorrect

Respond in JSON:
{{
  "hallucination_score": <0-5 where 5=none>,
  "claims": [
    {{
      "claim": "<from answer>",
      "status": "<SUPPORTED|UNCLEAR|UNSUPPORTED|HALLUCINATION>",
      "evidence": "<from reference or explanation>"
    }}
  ]
}}
"""

print("HALLUCINATION DETECTION PROMPT")
print("=" * 60)
# Fill with Example 2 (telephone, Margaret Thomson)
print(hallucination_prompt.format(
    reference=test_set[1]["ground_truth"],
    answer=test_set[1]["rag_response"]
))
print("Expected output would flag 'Margaret Thomson' as HALLUCINATION")
print("since this person does not appear in the reference or source documents.")

## 4.3 Advanced LLM Judge Techniques

### Technique 1: Chain-of-Thought Evaluation
Ask the LLM to **explain its reasoning before scoring**:
```
Before scoring, answer these:
1. What is the question asking for?
2. Does the answer address each part?
3. For each factual claim, is it verifiable?
4. What grade (0-5) for each dimension?
Then provide your scores.
```

### Technique 2: Comparison-Based Scoring
Instead of absolute 0-5 scales, **compare two answers**:
```
Answer A: "Python was created in 1989"
Answer B: "Python was designed in 1989 by Guido van Rossum"
Which is better? B wins (more complete)
```
Often **easier for LLMs** than absolute scoring.

### Technique 3: Multi-Step Evaluation
Break evaluation into **focused steps**:
1. **Fact Extraction:** "List all factual claims in the answer"
2. **Fact Checking:** "For each claim, mark SUPPORTED/UNSUPPORTED"
3. **Scoring:** "Now score on relevancy, accuracy, etc."

### Technique 4: Role-Specific Judges
Use **different prompts for different expertise**:
- Judge 1 (Fact Checker): "You are a researcher. Check facts."
- Judge 2 (Writing Coach): "You are an English professor. Check writing."
- Judge 3 (Domain Expert): "You are a [domain] expert. Check technical accuracy."

Then aggregate their scores.

## 4.4 Best Practices for LLM Judges

| Practice | Why |
|----------|-----|
| **Define scales explicitly** | "5 = all facts correct" not just "rate 0-5" |
| **Use structured output (JSON)** | Easy to parse and aggregate |
| **Include context** | Question + reference + sources |
| **Assign a role** | "You are an expert evaluator..." |
| **Give examples (few-shot)** | Show what 5/5 vs 1/5 looks like |
| **Use low temperature** | 0.3-0.5 for consistent scores |
| **Max 3-4 dimensions** | Too many confuses the LLM |
| **Handle uncertainty** | Ask for confidence scores |
| **Always validate** | Correlate with human labels (target > 0.7) |

---

# Part 5: General-Purpose vs. Specialized LLM Evaluators

## General-Purpose LLMs as Judges

Models like Claude, GPT-4, and Llama-70B can evaluate answers using **prompting alone** — no training required.

| Model | Speed | Cost/Eval | Quality | Best For |
|-------|-------|-----------|---------|----------|
| Claude Opus | Medium | ~\$0.01 | Excellent | Complex evaluation, medical/legal |
| Claude Sonnet | Fast | ~\$0.003 | Very Good | General screening |
| GPT-4 | Medium | ~\$0.01 | Excellent | Complex reasoning |
| GPT-4 Turbo | Fast | ~\$0.005 | Very Good | Balanced cost/quality |
| Llama-70B | Medium | Low (self-hosted) | Good | Open-source, privacy-sensitive |

### Advantages:
- Works immediately, no setup
- Can evaluate complex, nuanced scenarios
- Can explain its reasoning
- Flexible — can evaluate any dimension via prompting

### Disadvantages:
- Per-evaluation cost adds up at scale
- API-dependent (not offline)
- Not specialized to your domain
- Can hallucinate while evaluating
- Model updates can change scoring behavior

## Specialized/Fine-Tuned Evaluators

Small models (7-13B parameters) **fine-tuned specifically for evaluation tasks**.

### The ARES approach:
1. Generate synthetic training data from your documents
2. Fine-tune small models (Mistral-7B, DistilBERT) for binary classification
3. Use these specialized models for fast, cheap evaluation

### Advantages:
- 100-500x faster than LLM judges
- 100-1000x cheaper per evaluation
- Domain-specialized (trained on YOUR data)
- Runs offline, fully reproducible

### Disadvantages:
- Requires 1-2 weeks setup
- $1000-5000 upfront investment
- Less capable for complex/nuanced evaluation
- Needs retraining when documents change

## When to Choose Each

```
< 10,000 evaluations     → General-purpose LLM judges
10,000 - 100,000 evals   → Either approach works
> 100,000 evaluations    → Specialized/fine-tuned models
Real-time scoring needed → Only specialized models (10-100ms)
```

---

# Part 6: The RAGAS Framework

## What is RAGAS?

**RAGAS** (RAG Assessment) is an open-source framework that provides **pre-built evaluation metrics** specifically designed for RAG systems.

- **Creator:** Exploding Gradients (open-source team)
- **GitHub:** https://github.com/explodinggradients/ragas
- **Key feature:** Works out of the box with minimal setup

## RAGAS Core Metrics

RAGAS evaluates the **entire RAG pipeline** — both retrieval and generation:

```
        Question
            │
            ▼
     ┌───────────-──┐
     │   Retriever  │ ← Context Precision (are retrieved docs relevant?)
     │              │ ← Context Recall (are all needed facts retrieved?)
     └──────┬───────┘
            │
       Retrieved Docs
            │
            ▼
     ┌──────────-───┐
     │  Generator   │ ← Faithfulness (is answer grounded in context?)
     │   (LLM)      │ ← Answer Relevancy (does it answer the question?)
     └──────┬───────┘
            │
        Generated Answer
```

### Metric 1: Faithfulness (Answer Grounding)

**Question:** Does the answer only contain information from the retrieved context?

**How it works:**
1. LLM extracts claims from the generated answer
2. For each claim, LLM checks if it's supported by the retrieved context
3. Score = (supported claims) / (total claims)

**Score range:** 0.0 to 1.0
- 1.0 = All claims are supported by context (fully faithful)
- 0.0 = No claims are supported (pure hallucination)

**Example:**
```
Context: "Python was created in 1989 by Guido van Rossum"
Answer:  "Python was created in 1989 by Guido van Rossum at MIT"
Score:   0.67 — "at MIT" is NOT in the context (hallucination)
```

### Metric 2: Answer Relevancy

**Question:** Does the generated answer address the question?

**How it works (clever reverse-question approach):**
1. LLM generates 3-4 alternative questions that could produce the same answer
2. If the generated questions match the original question → answer is relevant
3. Score based on semantic similarity between original and generated questions

**Example:**
```
Original Q: "When was Python created?"
Answer:     "Python was created in 1989"
Generated Qs: "What year was Python released?" ✓
              "When did Python development start?" ✓
Score: 1.0 — all generated questions align with original
```

### Metric 3: Context Precision

**Question:** Of the retrieved documents, how many are actually relevant to the question?

$$\text{Context Precision} = \frac{\text{relevant retrieved docs}}{\text{total retrieved docs}}$$

Low precision = too much noise in retrieval.

### Metric 4: Context Recall

**Question:** Of the information needed to answer the question, how much is in the retrieved context?

$$\text{Context Recall} = \frac{\text{needed facts in context}}{\text{total facts needed}}$$

Low recall = important information is missing from retrieval.

### RAGAS Usage Example

```python
# pip install ragas

from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)

# Prepare your data
test_data = {
    "question": ["When was Python created?"],
    "answer": ["Python was created in 1989"],
    "contexts": [["Python was created in 1989 by Guido van Rossum"]],
    "ground_truth": ["1989"]
}

# Evaluate
results = evaluate(
    test_data,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall]
)

# Results
print(results)
# {
#   'faithfulness': 0.95,
#   'answer_relevancy': 0.92,
#   'context_precision': 1.0,
#   'context_recall': 1.0
# }
```

### RAGAS Strengths and Limitations

| Strengths | Limitations |
|-----------|-------------|
| Zero setup — works out of the box | LLM-dependent (costs per eval) |
| Excellent documentation | Not specialized to your domain |
| Pre-built metrics for RAG | Limited metric customization |
| Active community | Slower than fine-tuned models |
| Easy integration | Can't run offline (needs LLM API) |

---

# Part 7: The ARES Framework

## What is ARES?

**ARES** (Automated Retrieval Evaluation with Synthetic data) is a framework developed by Stanford researchers that takes a fundamentally different approach:

> Instead of paying for LLM evaluation every time, **train specialized models once** and evaluate cheaply forever.

**Paper:** "ARES: An Automated Evaluation Framework for Retrieval-Augmented Generation Systems" (Stanford, 2024)

## The Three Stages of ARES

```
┌────────────────────────────────────────────-──────┐
│ STAGE 1: SYNTHETIC DATA GENERATION (One-Time)     │
│                                                   │
│  Your Documents → LLM generates Q&A pairs         │
│  Create positive examples (good retrievals)       │
│  Create negative examples (bad retrievals)        │
│  Output: 1000-5000 labeled examples               │
│  Cost: ~$10-100 (one-time)                        │
└───────────────────────┬─────────────────────-─────┘
                        ↓
┌──────────────────────────────────────────────-────┐
│ STAGE 2: TRAIN EVALUATOR MODELS (One-Time)        │
│                                                   │
│  Fine-tune small models (Mistral-7B, DistilBERT)  │
│  Train 3 specialized evaluators:                  │
│    • Retrieval Evaluator (docs relevant to Q?)    │
│    • Answer Relevance Evaluator (A addresses Q?)  │
│    • Factuality Evaluator (A supported by docs?)  │
│  Cost: ~$500-1000 (compute for training)          │
└───────────────────────┬────────────────────────-──┘
                        ↓
┌─────────────────────────────────────────────────-─┐
│ STAGE 3: EVALUATE (Fast & Cheap — Repeated!)      │
│                                                   │
│  RAG outputs → Trained evaluators → Scores        │
│  Speed: 10-100ms per evaluation                   │
│  Cost: ~$0 per evaluation                         │
│  Can run offline                                  │
└────────────────────────────────────────────────-──┘
```

## Key Innovation

ARES applies the principle that works throughout ML:
- Computer vision: Train CNN **once**, use for millions of inferences
- NLP: Fine-tune BERT **once**, use for thousands of classifications
- **ARES: Train evaluators once**, evaluate at scale

## ARES Synthetic Data Generation

The data generation process creates training examples automatically:

```
Document: "Python was created in 1989 by Guido van Rossum"

Generated Questions:
  Q1: "When was Python created?"
  Q2: "Who created Python?"
  Q3: "What language did Guido van Rossum create?"

Training Examples:
  ✓ Positive: (Q1, Python_history_doc, "1989")   → RELEVANT
  ✗ Negative: (Q1, Java_programming_doc, "1989") → IRRELEVANT
```

## ARES Fine-Tuning

Each evaluator is a **binary classifier** trained on the synthetic data:

```python
# Pseudocode for ARES training
from transformers import AutoModelForSequenceClassification, Trainer

model = AutoModelForSequenceClassification.from_pretrained(
    "mistralai/Mistral-7B",
    num_labels=2  # Binary: relevant or not
)

trainer = Trainer(
    model=model,
    train_dataset=synthetic_data,
    args=TrainingArguments(
        learning_rate=2e-5,
        num_train_epochs=3,
    )
)
trainer.train()
```

Optimization techniques like **LoRA** (Low-Rank Adaptation) can reduce training cost by 90%.

## ARES Strengths and Limitations

| Strengths | Limitations |
|-----------|-------------|
| Domain-specialized (trained on YOUR docs) | Complex setup (1-2 weeks) |
| Extremely fast (10-100ms/eval) | Upfront cost ($1000-5000) |
| Extremely cheap at scale (~$0/eval) | Synthetic data may have biases |
| Runs offline, fully reproducible | Needs retraining when docs change |
| Deterministic output | Less capable for nuanced evaluation |

---

# Part 8: Comparing Evaluation Approaches

## Comprehensive Comparison Matrix

| Feature | Token Metrics | LLM Judge | RAGAS | ARES |
|---------|---------------|-----------|-------|------|
| **Setup Time** | None | Minutes | Minutes | 1-2 Weeks |
| **Setup Cost** | $0 | $0 | $0 | $1000-5000 |
| **Per-Eval Cost** | $0 | $0.001-0.01 | $0.001-0.01 | ~$0 |
| **Speed/Eval** | Instant | 1-5 sec | 1-10 sec | 10-100ms |
| **Accuracy** | 50-60% | 75-85% | 70-75% | 75-85% |
| **Domain-Specific** | No | No | Limited | Yes |
| **Runs Offline** | Yes | No | Partial | Yes |
| **Reproducible** | Yes | Partial | Partial | Yes |
| **Pre-built Metrics** | N/A | No | Yes | No |
| **Learning Curve** | Low | Low | Low | High |
| **Best For** | Prototypes | Quick eval | Learning | Production |

## Cost-Time-Accuracy Tradeoff

```
Method              | Cost (10k evals) | Time (10k evals) | Accuracy
─────────────────────────────────────────────────────────────────────
Token Metrics       | $0               | < 1 sec          | 50-60%
RAGAS               | $10-100          | 1-10 hrs         | 70-75%
LLM Judge           | $10-100          | 10-50 hrs        | 75-85%
ARES (post-setup)   | $0.10            | 2-5 min          | 75-85%
Human evaluation    | $5000+           | 50+ hrs          | 90-95%
```

## Decision Tree: Which Approach Should You Use?

```
START: "I need to evaluate my RAG system"
│
├─ Is this a prototype/POC?
│  YES → Token Metrics + RAGAS
│  Cost: $0, Time: hours
│
├─ < 10,000 evaluations?
│  YES → LLM Judges + RAGAS
│  Cost: $10-100, Time: days
│
├─ 10,000-100,000 evaluations?
│  YES → LLM Judges + human sampling
│  Cost: $100-500, Time: days
│
└─ > 100,000 evaluations?
   YES → ARES + LLM Judges for edge cases
   Cost: $2000-5000 setup, then ~$0
```

## The Hybrid Approach (Best Practice)

```
Tier 1: Fast Screening (ARES or RAGAS)
  → Score ALL answers
  → Cost: ~$0-10, Time: minutes
          │
    ┌─────┴───-───┐
    ▼             ▼
  Flagged      Good Items
  (Low scores)  → Accept
    │
    ├─ Tier 2: LLM Judge (20% of flagged)
    │  → Deeper analysis
    │  → Cost: $100-500
    │
    └─ Tier 3: Human Expert (5% of flagged)
       → Final validation
       → Cost: $500+

Result: 95-99% issue detection at 20% of LLM-only cost
```

---

# Part 9: Evaluating the Evaluation Frameworks

How do we know our evaluation metrics are **actually measuring what we think they're measuring**?

We evaluate the evaluators by examining:
1. **Score distributions** — Are scores well-distributed or clustered?
2. **Correlation with human judgment** — Do automated scores agree with human ratings?
3. **Cross-metric correlation** — Do different metrics agree with each other?

## Validation Strategy

1. Get human labels for ~100 examples
2. Compute automated scores on the same examples
3. Calculate correlation (Spearman's $\rho$ for ranking, Pearson's $r$ for absolute)
4. Target: correlation > 0.7 = reliable metric

In [15]:
import random
random.seed(42)

# Simulate evaluation scores from different methods
n_samples = 100

# Simulated human scores (ground truth)
human_relevancy = [random.gauss(3.5, 1.2) for _ in range(n_samples)]
human_relevancy = [max(0, min(5, round(s, 1))) for s in human_relevancy]

# Simulated automated scores (correlated with human but noisy)
def add_correlated_noise(base_scores, noise_level, correlation_target):
    """Create scores correlated with base scores."""
    noisy = []
    for s in base_scores:
        noise = random.gauss(0, noise_level)
        new_score = correlation_target * s + (1 - correlation_target) * random.gauss(3, 1) + noise
        noisy.append(max(0, min(5, round(new_score, 2))))
    return noisy

rouge_scores = add_correlated_noise(human_relevancy, 0.8, 0.5)  # Low correlation
llm_judge_scores = add_correlated_noise(human_relevancy, 0.3, 0.85)  # High correlation
ragas_scores = add_correlated_noise(human_relevancy, 0.5, 0.72)  # Medium correlation

print("Simulated Evaluation Scores (first 10 examples):")
print(f"{'#':<4} {'Human':<8} {'ROUGE':<8} {'LLM Judge':<10} {'RAGAS':<8}")
print("-" * 40)
for i in range(10):
    print(f"{i+1:<4} {human_relevancy[i]:<8.1f} {rouge_scores[i]:<8.2f} {llm_judge_scores[i]:<10.2f} {ragas_scores[i]:<8.2f}")

Simulated Evaluation Scores (first 10 examples):
#    Human    ROUGE    LLM Judge  RAGAS   
----------------------------------------
1    3.3      4.48     3.46       3.09    
2    3.3      2.17     3.29       2.93    
3    3.4      3.79     3.50       4.06    
4    4.3      4.65     4.17       4.54    
5    3.3      1.60     3.14       3.76    
6    1.7      2.52     2.00       2.41    
7    3.9      2.95     3.84       3.41    
8    3.2      3.95     2.98       2.63    
9    3.2      3.44     2.94       3.26    
10   3.6      2.76     4.19       4.09    


## 9.1 Analyzing Score Distributions

A good metric should produce **well-distributed scores** that discriminate between good and bad answers. If all scores cluster around one value, the metric is not useful.

In [16]:
def print_distribution(scores, name, bins=5):
    """Print a text histogram of score distribution."""
    bin_edges = [i for i in range(bins + 1)]
    counts = [0] * bins
    for s in scores:
        idx = min(int(s), bins - 1)
        counts[idx] += 1
    
    print(f"\n{name} Distribution (n={len(scores)}):")
    print(f"  Mean: {sum(scores)/len(scores):.2f}, Std: {(sum((s - sum(scores)/len(scores))**2 for s in scores)/len(scores))**0.5:.2f}")
    max_count = max(counts) if max(counts) > 0 else 1
    for i in range(bins):
        bar = '█' * int(counts[i] / max_count * 30)
        print(f"  [{i}-{i+1}): {counts[i]:>3} {bar}")

print("SCORE DISTRIBUTIONS")
print("=" * 60)
print_distribution(human_relevancy, "Human Scores")
print_distribution(rouge_scores, "ROUGE-based Scores")
print_distribution(llm_judge_scores, "LLM Judge Scores")
print_distribution(ragas_scores, "RAGAS Scores")

SCORE DISTRIBUTIONS

Human Scores Distribution (n=100):
  Mean: 3.53, Std: 0.97
  [0-1):   1 
  [1-2):   6 ████
  [2-3):  21 ████████████████
  [3-4):  33 █████████████████████████
  [4-5):  39 ██████████████████████████████

ROUGE-based Scores Distribution (n=100):
  Mean: 3.36, Std: 1.04
  [0-1):   1 
  [1-2):   8 ██████
  [2-3):  29 ████████████████████████
  [3-4):  35 ██████████████████████████████
  [4-5):  27 ███████████████████████

LLM Judge Scores Distribution (n=100):
  Mean: 3.43, Std: 0.84
  [0-1):   1 
  [1-2):   3 ██
  [2-3):  24 ████████████████
  [3-4):  45 ██████████████████████████████
  [4-5):  27 ██████████████████

RAGAS Scores Distribution (n=100):
  Mean: 3.33, Std: 0.81
  [0-1):   0 
  [1-2):   6 ███
  [2-3):  23 ██████████████
  [3-4):  49 ██████████████████████████████
  [4-5):  22 █████████████


## 9.2 Computing Correlations

The gold standard for evaluating metrics is **Spearman rank correlation** ($\rho$) with human judgment.

$$\rho = 1 - \frac{6 \sum d_i^2}{n(n^2 - 1)}$$

Where $d_i$ is the difference in ranks between metric score and human score.

**Interpretation:**
- $\rho > 0.8$: Excellent — can use metric with confidence
- $0.7 < \rho < 0.8$: Good — use with sampling validation
- $0.6 < \rho < 0.7$: Acceptable — needs more human oversight
- $\rho < 0.6$: Poor — revise approach

In [17]:
def spearman_correlation(x, y):
    """Compute Spearman rank correlation between two lists."""
    n = len(x)
    if n < 2:
        return 0.0
    
    def rank(data):
        sorted_indices = sorted(range(len(data)), key=lambda i: data[i])
        ranks = [0] * len(data)
        for rank_val, idx in enumerate(sorted_indices):
            ranks[idx] = rank_val + 1
        return ranks
    
    rank_x = rank(x)
    rank_y = rank(y)
    
    d_squared = sum((rx - ry) ** 2 for rx, ry in zip(rank_x, rank_y))
    rho = 1 - (6 * d_squared) / (n * (n**2 - 1))
    
    return rho

def pearson_correlation(x, y):
    """Compute Pearson correlation coefficient."""
    n = len(x)
    mean_x = sum(x) / n
    mean_y = sum(y) / n
    
    cov = sum((xi - mean_x) * (yi - mean_y) for xi, yi in zip(x, y)) / n
    std_x = (sum((xi - mean_x)**2 for xi in x) / n) ** 0.5
    std_y = (sum((yi - mean_y)**2 for yi in y) / n) ** 0.5
    
    if std_x == 0 or std_y == 0:
        return 0.0
    
    return cov / (std_x * std_y)

# Compute correlations with human scores
print("CORRELATION WITH HUMAN JUDGMENT")
print("=" * 70)
print(f"{'Metric':<20} {'Spearman ρ':<15} {'Pearson r':<15} {'Verdict':<20}")
print("-" * 70)

metrics = [
    ("ROUGE-based", rouge_scores),
    ("LLM Judge", llm_judge_scores),
    ("RAGAS", ragas_scores),
]

for name, scores in metrics:
    spearman = spearman_correlation(human_relevancy, scores)
    pearson = pearson_correlation(human_relevancy, scores)
    
    if spearman > 0.8:
        verdict = "Excellent ✓"
    elif spearman > 0.7:
        verdict = "Good ✓"
    elif spearman > 0.6:
        verdict = "Acceptable ⚠"
    else:
        verdict = "Poor ✗"
    
    print(f"{name:<20} {spearman:<15.3f} {pearson:<15.3f} {verdict:<20}")

print(f"\nTarget: Spearman ρ > 0.7 for reliable evaluation")

CORRELATION WITH HUMAN JUDGMENT
Metric               Spearman ρ      Pearson r       Verdict             
----------------------------------------------------------------------
ROUGE-based          0.356           0.375           Poor ✗              
LLM Judge            0.924           0.942           Excellent ✓         
RAGAS                0.686           0.745           Acceptable ⚠        

Target: Spearman ρ > 0.7 for reliable evaluation


## 9.3 Cross-Metric Agreement

We can also check if different automated metrics **agree with each other**. If they don't, some metrics may be measuring different things (or measuring nothing useful).

In [18]:
print("CROSS-METRIC CORRELATION MATRIX")
print("=" * 65)

all_metrics = {
    "Human": human_relevancy,
    "ROUGE": rouge_scores,
    "LLM Judge": llm_judge_scores,
    "RAGAS": ragas_scores,
}

names = list(all_metrics.keys())
header = f"{'':>12}" + "".join(f"{n:>12}" for n in names)
print(header)
print("-" * len(header))

for name_i in names:
    row = f"{name_i:>12}"
    for name_j in names:
        corr = spearman_correlation(all_metrics[name_i], all_metrics[name_j])
        row += f"{corr:>12.3f}"
    print(row)

print(f"\nKey observations:")
print(f"  • LLM Judge has highest correlation with Human scores")
print(f"  • ROUGE has lowest correlation — confirms its limitations")
print(f"  • RAGAS falls between ROUGE and LLM Judge")

CROSS-METRIC CORRELATION MATRIX
                   Human       ROUGE   LLM Judge       RAGAS
------------------------------------------------------------
       Human       1.000       0.356       0.924       0.686
       ROUGE       0.356       1.000       0.334       0.234
   LLM Judge       0.924       0.334       1.000       0.679
       RAGAS       0.686       0.234       0.679       1.000

Key observations:
  • LLM Judge has highest correlation with Human scores
  • ROUGE has lowest correlation — confirms its limitations
  • RAGAS falls between ROUGE and LLM Judge


## 9.4 Error Analysis: Where Metrics Disagree

The most informative analysis is examining cases where **metrics disagree with human judgment**.

In [ ]:
print("ERROR ANALYSIS: Cases where metrics most disagree with humans")
print("=" * 70)

# Find largest disagreements
disagreements = []
for i in range(n_samples):
    rouge_diff = abs(rouge_scores[i] - human_relevancy[i])
    llm_diff = abs(llm_judge_scores[i] - human_relevancy[i])
    disagreements.append((i, rouge_diff, llm_diff))

# Sort by ROUGE disagreement (largest first)
disagreements.sort(key=lambda x: x[1], reverse=True)

print(f"\nTop 5 cases where ROUGE disagrees most with Human:")
print(f"{'#':<4} {'Human':<8} {'ROUGE':<8} {'Diff':<8} {'LLM Judge':<10} {'LLM Diff':<8}")
print("-" * 50)
for idx, rouge_diff, llm_diff in disagreements[:5]:
    print(f"{idx+1:<4} {human_relevancy[idx]:<8.1f} {rouge_scores[idx]:<8.2f} "
          f"{rouge_diff:<8.2f} {llm_judge_scores[idx]:<10.2f} {llm_diff:<8.2f}")

print(f"\nPattern: ROUGE often has large disagreements (high error),")
print(f"while LLM Judge stays closer to human scores.")
print(f"\nThis confirms: Token metrics are unreliable as sole evaluators.")

## 9.5 Practical Validation Workflow

When deploying a RAG evaluation system, follow this workflow:

```
Step 1: Get human labels for ~100 examples
Step 2: Run all automated metrics on those 100 examples
Step 3: Compute Spearman correlation for each metric
Step 4: Decision:
        • ρ > 0.7  → Metric is reliable, use it
        • ρ < 0.7  → Revise prompt/approach, retry
Step 5: In production, periodically re-validate
        • Check for score drift over time
        • Flag outliers for manual review
```

---

# Summary and Key Takeaways

## The Evaluation Landscape

```
2022: Token metrics (ROUGE/BLEU)    → Fast but inaccurate
2023: LLM judges (Claude/GPT-4)     → Accurate but expensive
2024: Frameworks (RAGAS/ARES)        → Structured, scalable
2025: Hybrid systems                 → Best of all worlds
```

## Key Principles

1. **No single metric is sufficient** — Need multiple metrics across multiple dimensions

2. **Token metrics = Similarity, not Quality** — They measure if you match a reference, not if you're correct

3. **LLM judges bridge the gap** — Fast screening for relevancy, accuracy, fluency, and hallucinations

4. **Frameworks reduce complexity** — RAGAS for quick start, ARES for production scale

5. **Always validate with humans** — Compute correlation, target > 0.7

6. **The hybrid approach wins** — Automated screening + LLM judges for edge cases + human validation

## Recommended Evaluation Pipeline

| Layer | Method | Coverage | Cost |
|-------|--------|----------|------|
| **Layer 1** | Token metrics (ROUGE, BLEU) | All examples | Free |
| **Layer 2** | RAGAS/ARES framework | All examples | Low |
| **Layer 3** | LLM Judge (Claude/GPT-4) | Flagged + sample | Medium |
| **Layer 4** | Human evaluation | 5-10% sample | High |

## Final Decision Guide

| Your Situation | Recommendation |
|----------------|----------------|
| Quick prototype | Token metrics + RAGAS |
| < 10k evaluations | LLM Judges |
| 10k-100k evaluations | RAGAS + LLM Judges |
| > 100k evaluations | ARES + LLM Judges for edge cases |
| High-stakes domain | Full hybrid + human validation |